In [1]:
# kaggle API를 사용하기 위해 API 키 json 파일 업로드
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"jhw0714","key":"5ebfcd63c0dd4e608eb3f7b6fbbb5ebd"}'}

In [2]:
# kaggle.json 파일을 적절한 디렉토리로 옮긴다.
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# 데이터셋을 다운로드한다.
!kaggle datasets download -d vladimirvorobevv/chatgpt-paraphrases
!unzip -o /content/chatgpt-paraphrases.zip -d /content/sample_data

100% 60.0M/60.0M [00:04<00:00, 23.1MB/s]
100% 60.0M/60.0M [00:04<00:00, 15.3MB/s]
Archive:  /content/chatgpt-paraphrases.zip
  inflating: /content/sample_data/chatgpt_paraphrases.csv  


In [3]:
# 파이스파크 설치
!pip install pyspark
!pip install -U -q PyDrive
!apt install openjdk-8-jdk-headless -qq

# 한국어 텍스트 전처리를 도와주는 NLTK 패키지
!pip install nltk

import os
import re
from google.colab import drive
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
drive.mount('/content/drive')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.9/316.9 MB 3.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pyspark: filename=pyspark-3.5.0-py2.py3-none-any.whl size=317425344 sha256=904d260d6ec2013455789c32cb13afd431af11bfa5e7b988fbe85356fd3b1938
  Stored in directory: /root/.cache/pip/wheels/41/4e/10/c2cf2467f71c678cfc8a6b9ac9241e5e44a01940da8fbb17fc
Successfully built pyspark
The following additional packages will be installed:
  libxtst6 openjdk-8-jre-headless
Suggested packages:
  openjdk-8-demo openjdk-8-source libnss-mdns fonts-dejavu-extra fonts-nanum fonts-ipafont-gothic
  fonts-ipafont-mincho fonts-wqy-microhei fonts-wqy-zenhei fonts-indic
The following NEW packages will be installed:
  libxtst6 openjdk-8-jdk-headless openjdk-8-jre-headless
0 upgraded, 3 newly installed, 0 to remove and 10 not upgraded.
Need to get 39.7 MB of archives.
After this operation, 144 MB of additional disk space will be used.
Selecting previously unselected package

In [4]:
# Let's import the libraries we will need
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

import pyspark
from pyspark.sql import *
from pyspark.sql.functions import *
# SparkContext is the entry point to any spark functionality.
# SparkConf provides configurations to run a Spark application.
from pyspark import SparkContext, SparkConf

# create the Spark Session
spark = SparkSession.builder.appName("WordCount").getOrCreate()

# create the Spark Context
sc = spark.sparkContext

In [5]:
data=pd.read_csv('/content/sample_data/chatgpt_paraphrases.csv')

In [6]:
# 각 문장을 사람이 작성했는지, AI가 작성했는지 태그
category={}
for i in range(len(data)):
    chatgpt=data.iloc[i]["paraphrases"][1:-1].split(', ')
    for j in chatgpt[:1]:
        category[j[1:-1]]='chatgpt'
    category[data.iloc[i]['text']]="human"

In [7]:
# 사람의 작성한 문장과 AI가 작성한 문장을 분리
total_paraphrases=pd.DataFrame(category.items(),columns=["text","category"])
human_paraphrases = total_paraphrases[total_paraphrases['category'] == 'human']
gpt_paraphrases = total_paraphrases[total_paraphrases['category'] == 'chatgpt']

# 단어의 빈도 수 확인

In [8]:
import nltk

from nltk.corpus import stopwords
nltk.download('stopwords')

use_stopwords = True
english_stop_list = [".", ",", "!", "?", ")", "(", "'", '"', "]", "[", "{", "}", ":", ";", "<", ">", "~", "''", "``"]

if use_stopwords:
  english_stop_list += list(stopwords.words('english'))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [9]:
from nltk.stem import PorterStemmer
from nltk.stem import LancasterStemmer
from nltk.stem import RegexpStemmer
from nltk.stem import WordNetLemmatizer

from nltk.tokenize import sent_tokenize
from nltk.tokenize import word_tokenize

nltk.download('punkt')
nltk.download('wordnet')

lemmatizer = WordNetLemmatizer()

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


In [10]:
from nltk import pos_tag
from nltk.tokenize import word_tokenize
nltk.download('averaged_perceptron_tagger')
nltk.download('tagsets')

# 품사 매뉴얼 출력
nltk.help.upenn_tagset()

[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package tagsets to /root/nltk_data...


$: dollar
    $ -$ --$ A$ C$ HK$ M$ NZ$ S$ U.S.$ US$
'': closing quotation mark
    ' ''
(: opening parenthesis
    ( [ {
): closing parenthesis
    ) ] }
,: comma
    ,
--: dash
    --
.: sentence terminator
    . ! ?
:: colon or ellipsis
    : ; ...
CC: conjunction, coordinating
    & 'n and both but either et for less minus neither nor or plus so
    therefore times v. versus vs. whether yet
CD: numeral, cardinal
    mid-1890 nine-thirty forty-two one-tenth ten million 0.5 one forty-
    seven 1987 twenty '79 zero two 78-degrees eighty-four IX '60s .025
    fifteen 271,124 dozen quintillion DM2,000 ...
DT: determiner
    all an another any both del each either every half la many much nary
    neither no some such that the them these this those
EX: existential there
    there
FW: foreign word
    gemeinschaft hund ich jeux habeas Haementeria Herr K'ang-si vous
    lutihaw alai je jour objets salutaris fille quibusdam pas trop Monte
    terram fiche oui corporis ...
IN: preposition or

[nltk_data]   Unzipping help/tagsets.zip.


In [11]:
word_list_gpt = []
pos_list_gpt = []
for i in range(gpt_paraphrases.shape[0]):
  sentence_gpt = gpt_paraphrases.iloc[i]['text'].lower()
  sentence_tokens_gpt = word_tokenize(sentence_gpt)
  sentence_pos_gpt = pos_tag(sentence_tokens_gpt)
  for token_word in sentence_tokens_gpt:
    if token_word in english_stop_list:
      sentence_tokens_gpt.remove(token_word)

  pos_list_gpt += sentence_pos_gpt
  word_list_gpt += sentence_tokens_gpt

In [12]:
df_pos_gpt = {'pos': [pos[1] for pos in pos_list_gpt]}
pandas_df_pos_gpt = pd.DataFrame(data=df_pos_gpt)
spark_df_pos_gpt = spark.createDataFrame(pandas_df_pos_gpt)

all_pos_gpt = spark_df_pos_gpt.rdd.map(lambda poslist: (poslist.pos, 1))
all_pos_counts_gpt = all_pos_gpt.reduceByKey(lambda a, b: a+b).sortBy(lambda r: -r[1])
all_pos_counts_gpt.take(10)

[('NN', 1068050),
 ('IN', 575562),
 ('DT', 507743),
 ('JJ', 423269),
 ('NNS', 304373),
 ('.', 278256),
 ('VB', 205065),
 ('VBZ', 176750),
 ('TO', 136835),
 ('VBN', 110508)]

In [13]:
df_gpt = {'word': [lemmatizer.lemmatize(word , pos="v") for word in word_list_gpt]}
pandas_df_gpt = pd.DataFrame(data=df_gpt)
spark_df_gpt = spark.createDataFrame(pandas_df_gpt)

all_words_gpt = spark_df_gpt.rdd.map(lambda words: (words.word, 1))
all_words_counts_gpt = all_words_gpt.reduceByKey(lambda a, b: a+b).sortBy(lambda r: -r[1])
all_words_counts_gpt.take(10)

[('the', 126898),
 ('be', 120713),
 ('a', 57092),
 ("'s", 35939),
 ('you', 24212),
 ('it', 18519),
 ('do', 14347),
 ('reason', 13465),
 ('i', 12294),
 ('ways', 12025)]

In [14]:
word_list_hum = []
pos_list_hum = []
for i in range(human_paraphrases.shape[0]):
  sentence_hum = human_paraphrases.iloc[i]['text'].lower()
  sentence_tokens_hum = word_tokenize(sentence_hum)
  sentence_pos_hum = pos_tag(sentence_tokens_hum)
  for token_word in sentence_tokens_hum:
    if token_word in english_stop_list:
      sentence_tokens_hum.remove(token_word)

  pos_list_hum += sentence_pos_hum
  word_list_hum += sentence_tokens_hum

In [15]:
# 각 단어의 출현 횟수를 센다.
df_hum = {'word': [lemmatizer.lemmatize(word , pos="v") for word in word_list_hum]}
pandas_df_hum = pd.DataFrame(data=df_hum)
spark_df_hum = spark.createDataFrame(pandas_df_hum)

all_words_hum = spark_df_hum.rdd.map(lambda words: (words.word, 1))
all_words_counts_hum = all_words_hum.reduceByKey(lambda a, b: a+b).sortBy(lambda r: -r[1])
all_words_counts_hum.take(10)

[('the', 201229),
 ('be', 140846),
 ('a', 74958),
 ('do', 54234),
 ("'s", 39705),
 ('and', 28918),
 ('i', 21905),
 ('can', 21762),
 ('best', 19972),
 ('have', 19410)]

In [16]:
# 각 단어의 품사 갯수를 센다.
df_pos_hum = {'pos': [pos[1] for pos in pos_list_hum]}
pandas_df_pos_hum = pd.DataFrame(data=df_pos_hum)
spark_df_pos_hum = spark.createDataFrame(pandas_df_pos_hum)

all_pos_hum = spark_df_pos_hum.rdd.map(lambda poslist: (poslist.pos, 1))
all_pos_counts_hum = all_pos_hum.reduceByKey(lambda a, b: a+b).sortBy(lambda r: -r[1])
all_pos_counts_hum.take(10)

[('NN', 1491738),
 ('IN', 802471),
 ('DT', 640931),
 ('JJ', 622474),
 ('NNS', 448529),
 ('.', 447444),
 ('VB', 316633),
 (',', 240613),
 ('VBZ', 206327),
 ('RB', 205344)]

In [17]:
WordCount_Hum = all_words_counts_hum.toDF().toPandas().rename(columns = {'_1':'Word', '_2':'Count'  })
WordCount_GPT = all_words_counts_gpt.toDF().toPandas().rename(columns = {'_1':'Word', '_2':'Count'  })
PosCount_Hum = all_pos_counts_hum.toDF().toPandas().rename(columns = {'_1':'Position', '_2':'Count'  })
PosCount_GPT = all_pos_counts_gpt.toDF().toPandas().rename(columns = {'_1':'Position', '_2':'Count'  })

In [18]:
WordCount_Hum

,Word,Count
0,the,201229
1,be,140846
2,a,74958
3,do,54234
4,'s,39705
...,...,...
146024,appurtenances,1
146025,4:45,1
146026,post-operation,1
146027,dinah,1


In [19]:
WordCount_GPT

,Word,Count
0,the,126898
1,be,120713
2,a,57092
3,'s,35939
4,you,24212
...,...,...
112034,sisterhood,1
112035,confes,1
112036,mcnugget-sized,1
112037,thorning-schmid,1


In [20]:
PosCount_Hum

,Position,Count
0,NN,1491738
1,IN,802471
2,DT,640931
3,JJ,622474
4,NNS,448529
5,.,447444
6,VB,316633
7,",",240613
8,VBZ,206327
9,RB,205344


from matplotlib import pyplot as plt
PosCount_Hum['Count'].plot(kind='hist', bins=20, title='Count')
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
PosCount_Hum['Count'].plot(kind='line', figsize=(8, 4), title='Count')
plt.gca().spines[['top', 'right']].set_visible(False)

In [21]:
PosCount_GPT

,Position,Count
0,NN,1068050
1,IN,575562
2,DT,507743
3,JJ,423269
4,NNS,304373
5,.,278256
6,VB,205065
7,VBZ,176750
8,TO,136835
9,VBN,110508


from matplotlib import pyplot as plt
PosCount_GPT['Count'].plot(kind='hist', bins=20, title='Count')
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
PosCount_GPT['Count'].plot(kind='line', figsize=(8, 4), title='Count')
plt.gca().spines[['top', 'right']].set_visible(False)